[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srivtx/ai-anatomy/blob/main/notebooks/56_zoo_learning_paradigms.ipynb)

# 56 · Model zoo, part 7: the ways a model can learn
The same network can learn in very different ways depending on what signal it gets. Seven paradigms, each on a
small task you can run: supervised, semi-supervised (pseudo-labels), self-supervised (contrastive), transfer
learning, online learning, active learning, and reinforcement learning (a bandit and Q-learning).

**You're done when** all checks print ✅ with SOLUTIONS = False and you can say what signal each paradigm learns
from. CPU, ~20 minutes.

In [ ]:
import os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression, SGDClassifier
SOLUTIONS = os.environ.get("SOLUTIONS") == "1"; QUICK = os.environ.get("QUICK") == "1"
rng = np.random.default_rng(0); torch.manual_seed(0)
def check(name, got, want, tol=1e-3):
    if got is None or got is Ellipsis: print(f"⏳ {name}: not done yet"); return False
    try: ok = np.allclose(np.asarray(got, float), np.asarray(want, float), atol=tol)
    except Exception: ok = got == want
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f": got {got}, expected {want}")); return ok
d = load_digits(); X, y = d.data / 16, d.target
perm = rng.permutation(len(y)); X, y = X[perm], y[perm]; Xte, yte = X[1400:], y[1400:]; Xpool, ypool = X[:1400], y[:1400]
LAB = 50                                                    # pretend only 50 images are labelled
results = {}

## 1. Supervised: learn from (x, label) pairs
The baseline: train on the 50 labelled images only.

In [ ]:
sup = LogisticRegression(max_iter=3000).fit(Xpool[:LAB], ypool[:LAB]); results["supervised (50 labels)"] = sup.score(Xte, yte)
results["supervised (all 1,400 labels)"] = LogisticRegression(max_iter=3000).fit(Xpool, ypool).score(Xte, yte)

## 2. Semi-supervised: pseudo-labels
Use the model's own confident predictions on unlabelled data as extra labels, retrain, repeat. The signal comes
from the structure of the unlabelled data (similar images are probably the same digit).

In [ ]:
def pseudo_label_round(model, X_unlab, threshold=0.9):
    if SOLUTIONS:
        p = model.predict_proba(X_unlab); keep = p.max(1) >= threshold
        return X_unlab[keep], model.classes_[p.argmax(1)][keep]
    ...  # Exercise 1: return the unlabelled inputs the model is ≥ threshold confident about, with its predicted labels
model = sup; Xl, yl = Xpool[:LAB], ypool[:LAB]
for r in range(3):
    out = pseudo_label_round(model, Xpool[LAB:])
    if out is None: break
    Xl, yl = np.r_[Xpool[:LAB], out[0]], np.r_[ypool[:LAB], out[1]]; model = LogisticRegression(max_iter=3000).fit(Xl, yl)
    print(f"round {r + 1}: {len(out[1])} pseudo-labels, test accuracy {model.score(Xte, yte):.3f}")
results["semi-supervised: naive pseudo-labels"] = model.score(Xte, yte)
check("exercise 1: pseudo-labels returned", None if out is None else bool(len(out[1]) > 0 and set(out[1]) <= set(range(10))), True)

Naive pseudo-labelling usually *hurts* here: a model trained on 50 labels is confidently wrong on many images,
and retraining on its own mistakes locks them in (**confirmation bias**). Methods that use the *shape* of the
unlabelled data do better. **Label spreading** builds a similarity graph over all images and lets the 50 labels
flow along it to their neighbours.

In [ ]:
from sklearn.semi_supervised import LabelSpreading
ls_y = np.r_[ypool[:LAB], -np.ones(1400 - LAB, int)]               # -1 marks "unlabelled"
spread = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2).fit(Xpool, ls_y)
results["semi-supervised: label spreading"] = spread.score(Xte, yte)

## 3. Self-supervised: invent a task from the data itself
**Contrastive learning (SimCLR-style):** make two random distortions of each image; train an encoder so the two
views of the same image end up close and views of different images far apart. No labels used at all. Then
freeze the encoder and train a linear classifier on its features with only 50 labels.

In [ ]:
from scipy.ndimage import shift as nd_shift
def augment(x):
    imgs = x.reshape(-1, 8, 8); out = np.stack([nd_shift(im, rng.uniform(-1, 1, 2), order=1) for im in imgs])
    return torch.tensor(np.clip(out + rng.normal(0, 0.05, out.shape), 0, 1).reshape(len(x), -1), dtype=torch.float32)
def info_nce(z1, z2, tau=0.2):
    if SOLUTIONS:
        z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1); logits = z1 @ z2.T / tau
        return F.cross_entropy(logits, torch.arange(len(z1)))
    ...  # Exercise 2: normalise, similarity matrix / tau, each row's positive is on the diagonal → cross-entropy
check("exercise 2: InfoNCE is low when views match", None if info_nce(torch.eye(4), torch.eye(4)) is None else bool(info_nce(torch.eye(4), torch.eye(4)) < 0.05), True)
enc = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 64)); opt = torch.optim.Adam(enc.parameters(), 1e-3)
for step in range(150 if QUICK else 1500):
    idx = rng.choice(1400, 256, replace=False); a, b = augment(Xpool[idx]), augment(Xpool[idx])
    z1, z2 = F.normalize(enc(a), dim=1), F.normalize(enc(b), dim=1)
    loss = F.cross_entropy(z1 @ z2.T / 0.2, torch.arange(256)); opt.zero_grad(); loss.backward(); opt.step()
feats = lambda x: enc[:2](torch.tensor(x, dtype=torch.float32)).detach().numpy()
results["self-supervised features + 50 labels"] = LogisticRegression(max_iter=3000).fit(feats(Xpool[:LAB]), ypool[:LAB]).score(feats(Xte), yte)

## 4. Transfer learning: reuse what another task taught
Train a network on digits 0–4 (plenty of labels), then reuse its hidden layer for digits 5–9 with only a few
labels. Real-world version: start from a pretrained ImageNet or language model.

In [ ]:
src = ypool < 5; net = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 5)); opt = torch.optim.Adam(net.parameters(), 3e-3)
Xs, ys_ = torch.tensor(Xpool[src], dtype=torch.float32), torch.tensor(ypool[src])
for _ in range(100 if QUICK else 600): loss = F.cross_entropy(net(Xs), ys_); opt.zero_grad(); loss.backward(); opt.step()
tgt_tr = np.where(ypool >= 5)[0][:30]; tgt_te = yte >= 5
h = lambda x: net[:2](torch.tensor(x, dtype=torch.float32)).detach().numpy()
results["transfer: 30 labels of digits 5–9 (accuracy on 5–9)"] = LogisticRegression(max_iter=3000).fit(h(Xpool[tgt_tr]), ypool[tgt_tr]).score(h(Xte[tgt_te]), yte[tgt_te])
results["no transfer: same 30 labels (accuracy on 5–9)"] = LogisticRegression(max_iter=3000).fit(Xpool[tgt_tr], ypool[tgt_tr]).score(Xte[tgt_te], yte[tgt_te])

## 5. Online learning: one example at a time
Data arrives as a stream; update the model per example and never store the past. `partial_fit` does this.

In [ ]:
online = SGDClassifier(loss="log_loss", random_state=0); curve = []
for i in range(0, 1400, 20):
    online.partial_fit(Xpool[i:i + 20], ypool[i:i + 20], classes=np.arange(10)); curve.append(online.score(Xte, yte))
plt.plot(np.arange(20, 1420, 20), curve); plt.xlabel("examples seen"); plt.ylabel("test accuracy"); plt.title("online learning on a stream"); plt.show()
results["online (one pass over the stream)"] = curve[-1]

## 6. Active learning: ask for the labels that help most
Start with 10 labels; each round, ask a human to label the examples the model is least sure about.

In [ ]:
def most_uncertain(model, X_pool, k):
    if SOLUTIONS: return np.argsort(model.predict_proba(X_pool).max(1))[:k]
    ...  # Exercise 3: indices of the k pool examples with the lowest top-class probability
lab = list(range(10)); ok = True
for _ in range(4):
    m = LogisticRegression(max_iter=3000).fit(Xpool[lab], ypool[lab]); rest = np.setdiff1d(np.arange(1400), lab)
    pick = most_uncertain(m, Xpool[rest], 10)
    if pick is None: ok = False; break
    lab += list(rest[pick])
if ok:
    results["active learning (50 chosen labels)"] = LogisticRegression(max_iter=3000).fit(Xpool[lab], ypool[lab]).score(Xte, yte)
    check("exercise 3: active learning picks 40 new labels", len(lab), 50)
else: check("exercise 3", None, 50)

## 7. Reinforcement learning: learn from reward
No right answers, only a score for actions. **Bandit:** choose among slot machines; balance exploring and
exploiting (ε-greedy). **Q-learning:** in a small grid world, learn the value of each (state, action) from
Q(s,a) ← Q(s,a) + α [r + γ max Q(s′,·) − Q(s,a)].

In [ ]:
true_p = np.array([0.2, 0.5, 0.75, 0.4]); Qb, N = np.zeros(4), np.zeros(4); total = 0
for t in range(2000):
    a = rng.integers(4) if rng.random() < 0.1 else int(Qb.argmax()); r = rng.random() < true_p[a]
    N[a] += 1; Qb[a] += (r - Qb[a]) / N[a]; total += r
print(f"bandit: picked the best arm {N[2] / N.sum():.0%} of the time; average reward {total / 2000:.2f} (best possible 0.75)")

def q_update(Q, s, a, r, s2, alpha=0.5, gamma=0.9):
    if SOLUTIONS: Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a]); return Q
    ...  # Exercise 4: one Q-learning update (return Q)
Qt = q_update(np.zeros((2, 2)), 0, 1, 1.0, 1)
check("exercise 4: Q update", None if Qt is None else Qt[0, 1], 0.5)
if Qt is not None:
    W, H, goal = 5, 5, (4, 4); Q = np.zeros((W * H, 4)); moves = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    for ep in range(500):
        x, yy = 0, 0
        for _ in range(50):
            s = x * H + yy; a = rng.integers(4) if rng.random() < 0.2 else int(Q[s].argmax())
            x, yy = min(W - 1, max(0, x + moves[a][0])), min(H - 1, max(0, yy + moves[a][1])); r = 1.0 if (x, yy) == goal else -0.01
            Q = q_update(Q, s, a, r, x * H + yy)
            if (x, yy) == goal: break
    x, yy, steps = 0, 0, 0
    while (x, yy) != goal and steps < 30:
        a = int(Q[x * H + yy].argmax()); x, yy = min(W - 1, max(0, x + moves[a][0])), min(H - 1, max(0, yy + moves[a][1])); steps += 1
    print(f"Q-learning: greedy path reaches the goal in {steps} steps (shortest possible: 8)")

## The scoreboard

In [ ]:
for k, v in results.items(): print(f"{k:52s} {v:.3f}")

**What I measured, including the paradigms that didn't help here** (CPU, full run):

| Setup | Test accuracy |
|---|---|
| supervised, 50 labels | 0.839 |
| supervised, all 1,400 labels | 0.980 |
| naive pseudo-labels | 0.786 (worse: confirmation bias) |
| label spreading | **0.912** |
| contrastive features + 50 labels | 0.841 (no gain) |
| transfer from digits 0–4, 30 labels of 5–9 | 0.809 vs 0.902 without transfer (worse) |
| online, one pass | 0.919 |
| active learning, 50 chosen labels | 0.791 vs 0.839 for 50 random (worse) |

Why the failures are the lesson: **transfer** only helps when the source task's features are useful for the
target; a small net trained to tell 0–4 apart threw away what distinguishes 5–9. **Uncertainty sampling** from a
10-label start picks confusing outliers and skips whole classes (the cold-start problem); mixing in random picks
or clustering first fixes it. **Contrastive learning** needs augmentations that capture what doesn't matter, and
far more data and compute than this toy; at scale it's how vision foundation models are trained. Try the
stretch goals to turn each failure into a win.

### Exercise 5: name the paradigm
Values: "supervised", "semi_supervised", "self_supervised", "transfer", "online", "active", "reinforcement".

In [ ]:
def ex5():
    if SOLUTIONS: return {"pretraining_an_llm_on_next_token_prediction": "self_supervised", "fine_tuning_a_pretrained_vision_model": "transfer", "a_robot_learning_from_reward": "reinforcement",
                          "spam_filter_updated_with_every_new_email": "online", "doctors_label_only_the_scans_the_model_finds_hardest": "active"}
    ...
check("exercise 5", ex5(), {"pretraining_an_llm_on_next_token_prediction": "self_supervised", "fine_tuning_a_pretrained_vision_model": "transfer", "a_robot_learning_from_reward": "reinforcement",
                            "spam_filter_updated_with_every_new_email": "online", "doctors_label_only_the_scans_the_model_finds_hardest": "active"})

| Paradigm | The signal | Typical use |
|---|---|---|
| Supervised | a correct label per example | classification, regression |
| Semi-supervised | a few labels + the shape of unlabelled data | when labelling is expensive |
| Self-supervised | a task made from the data itself (next token, masked patch, matching views) | pretraining every foundation model |
| Transfer | knowledge from a related task | fine-tuning pretrained models |
| Online | a stream, one example at a time | recommendation, fraud, drifting data |
| Active | labels chosen by the model's uncertainty | expensive experts (medicine, law) |
| Reinforcement | reward for actions, often delayed | games, robots, RLHF and reasoning models |